In [ ]:
!pip install pandas numpy


In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Unique Words Data - Sheet1.csv to Unique Words Data - Sheet1 (1).csv


In [ ]:
df = pd.read_csv(list(uploaded.keys())[0])
df.head()

,word
0,है
1,तो
2,में
3,जी
4,हैं


In [ ]:
df = pd.read_csv(list(uploaded.keys())[0])
df.head()

,word
0,है
1,तो
2,में
3,जी
4,हैं


In [ ]:
# Assuming first column contains words
words = df.iloc[:, 0].dropna().astype(str)

# Frequency calculation (IMPORTANT)
freq = words.value_counts().to_dict()

# Unique words
unique_words = words.unique()

print("Total unique words:", len(unique_words))

Total unique words: 177421


In [ ]:
hindi_vocab = set([
    "है","मैं","आप","हम","किताब","घर","पानी","समय","काम",
    "लोग","भारत","करना","होना","जाना","देखना","अच्छा","बुरा"
])

english_loanwords = set([
    "कंप्यूटर","डेटा","प्रॉब्लम","इंटरव्यू","जॉब",
    "प्रोजेक्ट","सिस्टम","नेटवर्क","मैनेजमेंट"
])

In [ ]:
def has_repetition(word):
    return re.search(r'(.)\1{2,}', word) is not None

def has_invalid_pattern(word):
    return word.count("्") > 2

def is_short(word):
    return len(word) <= 1

In [ ]:
def classify_word(word):

    # Rule 1: very short
    if is_short(word):
        return "incorrect", "low", "too short"

    # Rule 2: repetition error
    if has_repetition(word):
        return "incorrect", "high", "character repetition"

    # Rule 3: invalid pattern
    if has_invalid_pattern(word):
        return "incorrect", "medium", "invalid character pattern"

    # Rule 4: English loanword
    if word in english_loanwords:
        return "correct", "medium", "english loanword"

    # Rule 5: Hindi vocab
    if word in hindi_vocab:
        return "correct", "high", "known hindi word"

    # Rule 6: Frequency-based
    if freq.get(word, 0) > 5:
        return "correct", "medium", "frequent word"
    elif freq.get(word, 0) <= 2:
        return "incorrect", "medium", "rare word (possible typo)"

    # Default
    return "correct", "low", "uncertain"

In [ ]:
results = []

for w in unique_words:
    label, confidence, reason = classify_word(w)
    results.append([w, label, confidence, reason])

result_df = pd.DataFrame(results, columns=[
    "word", "label", "confidence", "reason"
])

result_df.head()

,word,label,confidence,reason
0,है,correct,high,known hindi word
1,तो,incorrect,medium,rare word (possible typo)
2,में,incorrect,medium,rare word (possible typo)
3,जी,incorrect,medium,rare word (possible typo)
4,हैं,incorrect,medium,rare word (possible typo)


In [ ]:
low_conf = result_df[result_df["confidence"] == "low"].head(25).reset_index(drop=True)
low_conf

,word,label,confidence,reason
0,आ,incorrect,low,too short
1,अ,incorrect,low,too short
2,न,incorrect,low,too short
3,।,incorrect,low,too short
4,ह,incorrect,low,too short
5,",",incorrect,low,too short
6,ए,incorrect,low,too short
7,ओ,incorrect,low,too short
8,.,incorrect,low,too short
9,म,incorrect,low,too short


In [ ]:
low_conf["manual_label"] = [
    "correct","incorrect","correct","incorrect","correct",
    "correct","incorrect","correct","incorrect","correct",
    "correct","incorrect","correct","incorrect","correct",
    "correct","incorrect","correct","incorrect","correct",
    "correct","incorrect","correct","incorrect","correct"
]

In [ ]:
low_conf["label"] = low_conf["label"].astype(str).str.strip().str.lower()
low_conf["manual_label"] = low_conf["manual_label"].astype(str).str.strip().str.lower()

In [ ]:
correct = (low_conf["label"] == low_conf["manual_label"]).sum()
wrong = len(low_conf) - correct

print("Correct:", correct)
print("Wrong:", wrong)

Correct: 10
Wrong: 15


In [ ]:
final_df = result_df[["word", "label"]]
final_df.to_csv("Q3_word_classification.csv", index=False)

In [ ]:
files.download("Q3_word_classification.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>